# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_11 — C++ / Python Bindings with pybind11

### Research question

How can we expose a C++ finance/execution kernel to Python using `pybind11`, validate it against a Python reference implementation, benchmark performance, and package the interface in a research-friendly way?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
06_07_limit_board_margin_and_deadband_rebalancing
06_08_avellaneda_stoikov_market_making
06_09_limit_order_fill_probability_model
06_10_smart_order_router_simulator
06_11_cpp_python_bindings_pybind11
```

The earlier C++ notebook used a `ctypes` bridge. This notebook upgrades the bridge to `pybind11`, which gives a cleaner Python-facing API and native NumPy array support.

It covers:

1. why bindings matter;
2. `ctypes` versus `pybind11`;
3. Python reference implementation;
4. C++ kernel design;
5. NumPy array interface;
6. C++ exception handling;
7. compile-time extension building;
8. module import;
9. Python/C++ parity tests;
10. benchmark timing;
11. slippage and fill kernel;
12. rolling statistics kernel;
13. risk metric kernel;
14. memory-layout checks;
15. packaging layout;
16. fallback behaviour when `pybind11` is unavailable;
17. governance flags;
18. audit checks;
19. saved outputs and manifest.

The key idea:

> pybind11 lets a quant research repo keep Python ergonomics while moving hot loops into typed C++ kernels.

## 1. Why pybind11?

Python is ideal for:

- research iteration;
- notebooks;
- plotting;
- pandas workflows;
- model diagnostics.

C++ is useful for:

- low-latency loops;
- memory control;
- typed kernels;
- production translation;
- high-throughput simulation.

`pybind11` lets us write C++ functions and expose them as normal Python functions.

Unlike `ctypes`, `pybind11` can expose:

- C++ exceptions as Python exceptions;
- NumPy arrays as typed buffers;
- C++ classes;
- docstrings;
- overloaded functions;
- clean Python module interfaces.

The workflow is:

```text
Python reference → C++ kernel → pybind11 module → parity tests → benchmark → package
```

## 2. Binding design principles

A good C++/Python binding for quant research should be:

### Explicit

No hidden conversions or silent shape assumptions.

### Minimal

Keep the C++ kernel small and focused.

### Testable

Every C++ output should match a Python reference on deterministic inputs.

### Safe

Validate shapes, dtypes, finite values, and array contiguity.

### Portable

If no compiler or pybind11 installation is available, the notebook should fall back gracefully.

### Auditable

Save build logs, parity reports, benchmarks, and manifest.

## 3. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
import sysconfig
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class Pybind11BindingConfig:
    n_quotes: int = 200_000
    n_orders: int = 25_000
    seed: int = 42
    initial_mid: float = 100.0
    tick_size: float = 0.01
    base_depth: float = 2_500.0
    taker_fee_bps: float = 0.45
    slippage_bps: float = 0.20
    latency_events: int = 2
    rolling_window: int = 64
    benchmark_repeats: int = 5
    output_subdir: str = "cpp_python_bindings_pybind11_v1"
    module_name: str = "quant_cpp_kernels"

config = Pybind11BindingConfig()
config

## 4. Simulate quote and order arrays

The kernel will process plain NumPy arrays.

We simulate:

- bid;
- ask;
- mid;
- bid size;
- ask size;
- order side;
- quantity;
- submission index.

The C++ kernel will return:

- fill price;
- filled quantity;
- fee;
- shortfall in basis points.

In [ ]:
def round_to_tick(x, tick):
    return np.round(x / tick) * tick

def simulate_quotes_and_orders(config: Pybind11BindingConfig):
    rng = np.random.default_rng(config.seed)

    n = config.n_quotes
    returns = rng.standard_t(df=6, size=n) * 0.00012
    shock_idx = rng.choice(n, size=max(5, n // 10_000), replace=False)
    returns[shock_idx] += rng.normal(0.0, 0.0025, size=len(shock_idx))

    mid = config.initial_mid * np.exp(np.cumsum(returns))
    mid = round_to_tick(mid, config.tick_size)

    spread_ticks = rng.choice(np.arange(1, 8), size=n, p=[0.42, 0.25, 0.13, 0.08, 0.05, 0.04, 0.03])
    spread = spread_ticks * config.tick_size

    bid = round_to_tick(mid - spread / 2.0, config.tick_size)
    ask = round_to_tick(mid + spread / 2.0, config.tick_size)
    ask = np.maximum(ask, bid + config.tick_size)
    mid = (bid + ask) / 2.0

    bid_size = rng.lognormal(np.log(config.base_depth), 0.60, size=n)
    ask_size = rng.lognormal(np.log(config.base_depth), 0.60, size=n)

    n_orders = config.n_orders
    submit_idx = np.sort(rng.choice(np.arange(0, n - config.latency_events - 1), size=n_orders, replace=False))
    side = rng.choice(np.array([1, -1], dtype=np.int32), size=n_orders)
    qty = rng.lognormal(np.log(800), 0.70, size=n_orders)
    qty = np.maximum(1.0, np.round(qty))

    arrival_mid = mid[submit_idx]

    quotes = pd.DataFrame({
        "bid": bid,
        "ask": ask,
        "mid": mid,
        "bid_size": bid_size,
        "ask_size": ask_size,
    })

    orders = pd.DataFrame({
        "submit_idx": submit_idx.astype(np.int64),
        "side": side.astype(np.int32),
        "qty": qty.astype(float),
        "arrival_mid": arrival_mid.astype(float),
    })

    return quotes, orders

quotes, orders = simulate_quotes_and_orders(config)

quotes.head(), orders.head()

## 5. Python reference fill kernel

This function is the ground truth for parity testing.

For each order:

1. apply latency;
2. buy order fills at ask;
3. sell order fills at bid;
4. fill is capped by top-of-book visible size;
5. fee is taker fee plus slippage;
6. shortfall is measured against arrival mid.

In [ ]:
def python_market_fill_kernel(
    bid,
    ask,
    bid_size,
    ask_size,
    submit_idx,
    side,
    qty,
    arrival_mid,
    latency_events,
    taker_fee_bps,
    slippage_bps,
):
    n_orders = len(qty)

    fill_price = np.empty(n_orders, dtype=np.float64)
    fill_qty = np.empty(n_orders, dtype=np.float64)
    fee = np.empty(n_orders, dtype=np.float64)
    shortfall_bps = np.empty(n_orders, dtype=np.float64)

    n_quotes = len(bid)

    for i in range(n_orders):
        idx = int(submit_idx[i]) + int(latency_events)
        if idx < 0:
            idx = 0
        if idx >= n_quotes:
            idx = n_quotes - 1

        if side[i] > 0:
            px = ask[idx]
            available = ask_size[idx]
            filled = min(qty[i], available)
            sf = (px - arrival_mid[i]) / arrival_mid[i] * 10000.0
        else:
            px = bid[idx]
            available = bid_size[idx]
            filled = min(qty[i], available)
            sf = (arrival_mid[i] - px) / arrival_mid[i] * 10000.0

        total_fee_bps = taker_fee_bps + slippage_bps
        f = filled * px * total_fee_bps / 10000.0
        sf += total_fee_bps

        fill_price[i] = px
        fill_qty[i] = filled
        fee[i] = f
        shortfall_bps[i] = sf

    return {
        "fill_price": fill_price,
        "fill_qty": fill_qty,
        "fee": fee,
        "shortfall_bps": shortfall_bps,
    }

py_fills = python_market_fill_kernel(
    quotes["bid"].to_numpy(),
    quotes["ask"].to_numpy(),
    quotes["bid_size"].to_numpy(),
    quotes["ask_size"].to_numpy(),
    orders["submit_idx"].to_numpy(),
    orders["side"].to_numpy(),
    orders["qty"].to_numpy(),
    orders["arrival_mid"].to_numpy(),
    config.latency_events,
    config.taker_fee_bps,
    config.slippage_bps,
)

pd.DataFrame(py_fills).head()

## 6. Python reference rolling statistics kernel

A second kernel computes rolling mean and rolling standard deviation.

This demonstrates that pybind11 can expose more than execution functions.

In [ ]:
def python_rolling_mean_std(x, window):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    mean = np.full(n, np.nan)
    std = np.full(n, np.nan)

    for i in range(window - 1, n):
        chunk = x[i - window + 1:i + 1]
        mean[i] = chunk.mean()
        std[i] = chunk.std(ddof=1)

    return {"mean": mean, "std": std}

returns = quotes["mid"].pct_change().fillna(0.0).to_numpy()
py_rolling = python_rolling_mean_std(returns, config.rolling_window)

pd.DataFrame(py_rolling).tail()

## 7. Python reference risk metrics

A third kernel computes:

- mean;
- volatility;
- downside volatility;
- maximum drawdown;
- hit rate.

This shows how a C++ kernel can return a compact metric vector.

In [ ]:
def python_risk_metrics(returns):
    r = np.asarray(returns, dtype=np.float64)
    nav = np.cumprod(1.0 + r)
    dd = nav / np.maximum.accumulate(nav) - 1.0
    downside = r[r < 0]

    return np.array([
        r.mean(),
        r.std(ddof=1),
        downside.std(ddof=1) if len(downside) > 1 else 0.0,
        dd.min(),
        (r > 0).mean(),
    ], dtype=np.float64)

py_risk = python_risk_metrics(returns)

pd.Series(py_risk, index=["mean", "vol", "downside_vol", "max_drawdown", "hit_rate"])

## 8. C++ pybind11 source

The C++ module exposes three functions:

1. `market_fill_kernel`
2. `rolling_mean_std`
3. `risk_metrics`

It uses `py::array_t<T>` for NumPy arrays and performs explicit shape checks.

If input shapes are invalid, the C++ function throws an exception, which appears as a Python exception.

In [ ]:
cpp_source = r"""
#include <pybind11/pybind11.h>
#include <pybind11/numpy.h>
#include <cmath>
#include <stdexcept>
#include <algorithm>
#include <limits>

namespace py = pybind11;

py::dict market_fill_kernel(
    py::array_t<double, py::array::c_style | py::array::forcecast> bid,
    py::array_t<double, py::array::c_style | py::array::forcecast> ask,
    py::array_t<double, py::array::c_style | py::array::forcecast> bid_size,
    py::array_t<double, py::array::c_style | py::array::forcecast> ask_size,
    py::array_t<long long, py::array::c_style | py::array::forcecast> submit_idx,
    py::array_t<int, py::array::c_style | py::array::forcecast> side,
    py::array_t<double, py::array::c_style | py::array::forcecast> qty,
    py::array_t<double, py::array::c_style | py::array::forcecast> arrival_mid,
    int latency_events,
    double taker_fee_bps,
    double slippage_bps
) {
    auto b = bid.unchecked<1>();
    auto a = ask.unchecked<1>();
    auto bs = bid_size.unchecked<1>();
    auto as = ask_size.unchecked<1>();
    auto sub = submit_idx.unchecked<1>();
    auto sd = side.unchecked<1>();
    auto q = qty.unchecked<1>();
    auto am = arrival_mid.unchecked<1>();

    const ssize_t n_quotes = b.shape(0);
    if (a.shape(0) != n_quotes || bs.shape(0) != n_quotes || as.shape(0) != n_quotes) {
        throw std::invalid_argument("Quote arrays must have identical length.");
    }

    const ssize_t n_orders = q.shape(0);
    if (sub.shape(0) != n_orders || sd.shape(0) != n_orders || am.shape(0) != n_orders) {
        throw std::invalid_argument("Order arrays must have identical length.");
    }

    py::array_t<double> fill_price(n_orders);
    py::array_t<double> fill_qty(n_orders);
    py::array_t<double> fee(n_orders);
    py::array_t<double> shortfall_bps(n_orders);

    auto out_px = fill_price.mutable_unchecked<1>();
    auto out_qty = fill_qty.mutable_unchecked<1>();
    auto out_fee = fee.mutable_unchecked<1>();
    auto out_sf = shortfall_bps.mutable_unchecked<1>();

    for (ssize_t i = 0; i < n_orders; ++i) {
        long long idx = sub(i) + latency_events;
        if (idx < 0) idx = 0;
        if (idx >= n_quotes) idx = n_quotes - 1;

        double px = 0.0;
        double available = 0.0;
        double sf = 0.0;

        if (sd(i) > 0) {
            px = a(idx);
            available = as(idx);
            sf = (px - am(i)) / am(i) * 10000.0;
        } else {
            px = b(idx);
            available = bs(idx);
            sf = (am(i) - px) / am(i) * 10000.0;
        }

        double filled = std::min(q(i), std::max(available, 0.0));
        double total_fee_bps = taker_fee_bps + slippage_bps;
        double f = filled * px * total_fee_bps / 10000.0;
        sf += total_fee_bps;

        out_px(i) = px;
        out_qty(i) = filled;
        out_fee(i) = f;
        out_sf(i) = sf;
    }

    py::dict result;
    result["fill_price"] = fill_price;
    result["fill_qty"] = fill_qty;
    result["fee"] = fee;
    result["shortfall_bps"] = shortfall_bps;
    return result;
}

py::dict rolling_mean_std(
    py::array_t<double, py::array::c_style | py::array::forcecast> x,
    int window
) {
    if (window <= 1) {
        throw std::invalid_argument("window must be greater than 1.");
    }

    auto arr = x.unchecked<1>();
    const ssize_t n = arr.shape(0);

    py::array_t<double> mean(n);
    py::array_t<double> out_std_array(n);
    auto out_mean = mean.mutable_unchecked<1>();
    auto out_std = out_std_array.mutable_unchecked<1>();

    const double nan = std::numeric_limits<double>::quiet_NaN();

    for (ssize_t i = 0; i < n; ++i) {
        out_mean(i) = nan;
        out_std(i) = nan;
    }

    double sum = 0.0;
    double sumsq = 0.0;

    for (ssize_t i = 0; i < n; ++i) {
        double val = arr(i);
        sum += val;
        sumsq += val * val;

        if (i >= window) {
            double old = arr(i - window);
            sum -= old;
            sumsq -= old * old;
        }

        if (i >= window - 1) {
            double m = sum / window;
            double var = (sumsq - window * m * m) / (window - 1);
            if (var < 0.0 && var > -1e-18) var = 0.0;
            out_mean(i) = m;
            out_std(i) = std::sqrt(std::max(var, 0.0));
        }
    }

    py::dict result;
    result["mean"] = mean;
    result["std"] = out_std_array;
    return result;
}

py::array_t<double> risk_metrics(
    py::array_t<double, py::array::c_style | py::array::forcecast> returns
) {
    auto r = returns.unchecked<1>();
    const ssize_t n = r.shape(0);

    if (n < 2) {
        throw std::invalid_argument("returns must have at least two observations.");
    }

    double sum = 0.0;
    double sumsq = 0.0;
    double downside_sum = 0.0;
    double downside_sumsq = 0.0;
    int downside_n = 0;
    int positive_n = 0;

    double nav = 1.0;
    double peak = 1.0;
    double max_dd = 0.0;

    for (ssize_t i = 0; i < n; ++i) {
        double x = r(i);
        sum += x;
        sumsq += x * x;
        if (x < 0.0) {
            downside_sum += x;
            downside_sumsq += x * x;
            downside_n += 1;
        }
        if (x > 0.0) positive_n += 1;

        nav *= (1.0 + x);
        if (nav > peak) peak = nav;
        double dd = nav / peak - 1.0;
        if (dd < max_dd) max_dd = dd;
    }

    double mean = sum / n;
    double var = (sumsq - n * mean * mean) / (n - 1);
    double vol = std::sqrt(std::max(var, 0.0));

    double downside_vol = 0.0;
    if (downside_n > 1) {
        double dm = downside_sum / downside_n;
        double dvar = (downside_sumsq - downside_n * dm * dm) / (downside_n - 1);
        downside_vol = std::sqrt(std::max(dvar, 0.0));
    }

    double hit_rate = static_cast<double>(positive_n) / n;

    py::array_t<double> out(5);
    auto o = out.mutable_unchecked<1>();
    o(0) = mean;
    o(1) = vol;
    o(2) = downside_vol;
    o(3) = max_dd;
    o(4) = hit_rate;

    return out;
}

PYBIND11_MODULE(quant_cpp_kernels, m) {
    m.doc() = "C++ kernels for execution and risk analytics exposed with pybind11.";

    m.def(
        "market_fill_kernel",
        &market_fill_kernel,
        "Marketable order fill kernel using L1 bid/ask arrays."
    );

    m.def(
        "rolling_mean_std",
        &rolling_mean_std,
        "Rolling mean and standard deviation kernel."
    );

    m.def(
        "risk_metrics",
        &risk_metrics,
        "Risk metric vector: mean, volatility, downside volatility, max drawdown, hit rate."
    );
}
"""

output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

cpp_path = output_dir / f"{config.module_name}.cpp"
cpp_path.write_text(cpp_source, encoding="utf-8")

cpp_path

## 9. Check pybind11 availability

The notebook can only build the extension if:

1. a C++ compiler is available;
2. `pybind11` is installed;
3. Python development headers are available.

If not, the notebook records the issue and continues with Python-only fallback.

In [ ]:
def check_build_environment():
    compiler = shutil.which("c++") or shutil.which("g++") or shutil.which("clang++")
    try:
        import pybind11
        pybind11_available = True
        pybind11_include = pybind11.get_include()
    except Exception:
        pybind11_available = False
        pybind11_include = None

    extension_suffix = sysconfig.get_config_var("EXT_SUFFIX")
    python_include = sysconfig.get_paths().get("include")

    return {
        "compiler": compiler,
        "pybind11_available": pybind11_available,
        "pybind11_include": pybind11_include,
        "extension_suffix": extension_suffix,
        "python_include": python_include,
        "platform": platform.platform(),
        "python_version": sys.version,
    }

build_env = check_build_environment()
build_env

## 10. Build pybind11 extension

The build command is intentionally explicit.

On Unix-like systems it resembles:

```bash
c++ -O3 -Wall -shared -std=c++17 -fPIC \
    $(python -m pybind11 --includes) \
    quant_cpp_kernels.cpp \
    -o quant_cpp_kernels$(python-config --extension-suffix)
```

On unsupported environments, the notebook falls back gracefully.

In [ ]:
def build_pybind11_extension(cpp_path: Path, module_name: str, build_env: dict):
    if build_env["compiler"] is None:
        return None, "No C++ compiler found."

    if not build_env["pybind11_available"]:
        return None, "pybind11 is not installed."

    if build_env["extension_suffix"] is None:
        return None, "Could not determine Python extension suffix."

    if build_env["python_include"] is None:
        return None, "Could not determine Python include path."

    compiler = build_env["compiler"]
    output_path = cpp_path.parent / f"{module_name}{build_env['extension_suffix']}"

    cmd = [
        compiler,
        "-O3",
        "-Wall",
        "-shared",
        "-std=c++17",
        "-fPIC",
        f"-I{build_env['python_include']}",
        f"-I{build_env['pybind11_include']}",
        str(cpp_path),
        "-o",
        str(output_path),
    ]

    if platform.system().lower() == "darwin":
        cmd.insert(-2, "-undefined")
        cmd.insert(-2, "dynamic_lookup")

    result = subprocess.run(cmd, capture_output=True, text=True)

    build_log = {
        "cmd": " ".join(cmd),
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "output_path": str(output_path),
    }

    if result.returncode != 0:
        return None, build_log

    return output_path, build_log

extension_path, build_log = build_pybind11_extension(cpp_path, config.module_name, build_env)

extension_path, build_log if isinstance(build_log, str) else build_log["returncode"]

## 11. Import compiled module

If the extension compiled, add the build folder to `sys.path` and import the module.

In [ ]:
def import_cpp_module(extension_path, module_name):
    if extension_path is None:
        return None, "Extension unavailable."

    module_dir = str(Path(extension_path).parent.resolve())
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)

    try:
        if module_name in sys.modules:
            del sys.modules[module_name]
        module = importlib.import_module(module_name)
        return module, "imported"
    except Exception as exc:
        return None, repr(exc)

cpp_module, import_message = import_cpp_module(extension_path, config.module_name)

cpp_module, import_message

## 12. Run C++ kernels if available

The outputs should be NumPy arrays inside Python dictionaries.

In [ ]:
def run_cpp_kernels(cpp_module, quotes, orders, returns, config):
    if cpp_module is None:
        return None, None, None

    cpp_fills = cpp_module.market_fill_kernel(
        np.ascontiguousarray(quotes["bid"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(quotes["ask"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(quotes["bid_size"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(quotes["ask_size"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(orders["submit_idx"].to_numpy(dtype=np.int64)),
        np.ascontiguousarray(orders["side"].to_numpy(dtype=np.int32)),
        np.ascontiguousarray(orders["qty"].to_numpy(dtype=np.float64)),
        np.ascontiguousarray(orders["arrival_mid"].to_numpy(dtype=np.float64)),
        int(config.latency_events),
        float(config.taker_fee_bps),
        float(config.slippage_bps),
    )

    cpp_rolling = cpp_module.rolling_mean_std(
        np.ascontiguousarray(returns.astype(np.float64)),
        int(config.rolling_window),
    )

    cpp_risk = cpp_module.risk_metrics(
        np.ascontiguousarray(returns.astype(np.float64))
    )

    return cpp_fills, cpp_rolling, cpp_risk

cpp_fills, cpp_rolling, cpp_risk = run_cpp_kernels(cpp_module, quotes, orders, returns, config)

(None if cpp_fills is None else pd.DataFrame({k: np.asarray(v) for k, v in cpp_fills.items()}).head())

## 13. Parity tests

C++ must match Python reference before it is trusted.

We check:

- fill price;
- filled quantity;
- fee;
- shortfall;
- rolling mean;
- rolling std;
- risk metrics.

In [ ]:
def max_abs_diff(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    mask = ~(np.isnan(a) & np.isnan(b))
    if mask.sum() == 0:
        return 0.0
    return float(np.nanmax(np.abs(a[mask] - b[mask])))

def build_parity_report(py_fills, py_rolling, py_risk, cpp_fills, cpp_rolling, cpp_risk):
    rows = []

    if cpp_fills is None:
        return pd.DataFrame([{
            "kernel": "all",
            "metric": "cpp_available",
            "max_abs_diff": np.nan,
            "tolerance": np.nan,
            "passed": False,
            "note": "C++ module unavailable; Python fallback only.",
        }])

    for key in ["fill_price", "fill_qty", "fee", "shortfall_bps"]:
        diff = max_abs_diff(py_fills[key], np.asarray(cpp_fills[key]))
        rows.append({
            "kernel": "market_fill_kernel",
            "metric": key,
            "max_abs_diff": diff,
            "tolerance": 1e-9,
            "passed": bool(diff < 1e-9),
            "note": "Python/C++ parity",
        })

    for key in ["mean", "std"]:
        diff = max_abs_diff(py_rolling[key], np.asarray(cpp_rolling[key]))
        rows.append({
            "kernel": "rolling_mean_std",
            "metric": key,
            "max_abs_diff": diff,
            "tolerance": 1e-10,
            "passed": bool(diff < 1e-10),
            "note": "Python/C++ parity",
        })

    diff = max_abs_diff(py_risk, np.asarray(cpp_risk))
    rows.append({
        "kernel": "risk_metrics",
        "metric": "metric_vector",
        "max_abs_diff": diff,
        "tolerance": 1e-10,
        "passed": bool(diff < 1e-10),
        "note": "Python/C++ parity",
    })

    return pd.DataFrame(rows)

parity_report = build_parity_report(py_fills, py_rolling, py_risk, cpp_fills, cpp_rolling, cpp_risk)

parity_report

## 14. Exception handling test

A binding should fail loudly on invalid input.

We intentionally pass mismatched array lengths to verify that C++ throws a Python-visible exception.

In [ ]:
def exception_handling_test(cpp_module, quotes, orders, config):
    if cpp_module is None:
        return pd.DataFrame([{
            "test": "mismatched_quote_lengths",
            "passed": False,
            "message": "C++ module unavailable.",
        }])

    try:
        cpp_module.market_fill_kernel(
            np.ascontiguousarray(quotes["bid"].to_numpy(dtype=np.float64)[:-1]),
            np.ascontiguousarray(quotes["ask"].to_numpy(dtype=np.float64)),
            np.ascontiguousarray(quotes["bid_size"].to_numpy(dtype=np.float64)),
            np.ascontiguousarray(quotes["ask_size"].to_numpy(dtype=np.float64)),
            np.ascontiguousarray(orders["submit_idx"].to_numpy(dtype=np.int64)),
            np.ascontiguousarray(orders["side"].to_numpy(dtype=np.int32)),
            np.ascontiguousarray(orders["qty"].to_numpy(dtype=np.float64)),
            np.ascontiguousarray(orders["arrival_mid"].to_numpy(dtype=np.float64)),
            int(config.latency_events),
            float(config.taker_fee_bps),
            float(config.slippage_bps),
        )
        return pd.DataFrame([{
            "test": "mismatched_quote_lengths",
            "passed": False,
            "message": "No exception raised.",
        }])
    except Exception as exc:
        return pd.DataFrame([{
            "test": "mismatched_quote_lengths",
            "passed": True,
            "message": repr(exc),
        }])

exception_report = exception_handling_test(cpp_module, quotes, orders, config)

exception_report

## 15. Benchmark timing

We compare:

- Python reference fill kernel;
- C++ pybind11 fill kernel;
- Python rolling statistics;
- C++ rolling statistics;
- Python risk metrics;
- C++ risk metrics.

Notebook timings are approximate. Use them for direction, not as production benchmarking.

In [ ]:
def benchmark_pybind11(cpp_module, quotes, orders, returns, config):
    rows = []

    for repeat in range(config.benchmark_repeats):
        start = time.perf_counter()
        _ = python_market_fill_kernel(
            quotes["bid"].to_numpy(),
            quotes["ask"].to_numpy(),
            quotes["bid_size"].to_numpy(),
            quotes["ask_size"].to_numpy(),
            orders["submit_idx"].to_numpy(),
            orders["side"].to_numpy(),
            orders["qty"].to_numpy(),
            orders["arrival_mid"].to_numpy(),
            config.latency_events,
            config.taker_fee_bps,
            config.slippage_bps,
        )
        elapsed = time.perf_counter() - start
        rows.append({
            "kernel": "market_fill_kernel",
            "engine": "python",
            "repeat": repeat,
            "elapsed_seconds": elapsed,
            "items_per_second": len(orders) / elapsed,
        })

    for repeat in range(config.benchmark_repeats):
        start = time.perf_counter()
        _ = python_rolling_mean_std(returns, config.rolling_window)
        elapsed = time.perf_counter() - start
        rows.append({
            "kernel": "rolling_mean_std",
            "engine": "python",
            "repeat": repeat,
            "elapsed_seconds": elapsed,
            "items_per_second": len(returns) / elapsed,
        })

    for repeat in range(config.benchmark_repeats):
        start = time.perf_counter()
        _ = python_risk_metrics(returns)
        elapsed = time.perf_counter() - start
        rows.append({
            "kernel": "risk_metrics",
            "engine": "python",
            "repeat": repeat,
            "elapsed_seconds": elapsed,
            "items_per_second": len(returns) / elapsed,
        })

    if cpp_module is not None:
        _ = run_cpp_kernels(cpp_module, quotes, orders, returns, config)

        for repeat in range(config.benchmark_repeats):
            start = time.perf_counter()
            _ = cpp_module.market_fill_kernel(
                np.ascontiguousarray(quotes["bid"].to_numpy(dtype=np.float64)),
                np.ascontiguousarray(quotes["ask"].to_numpy(dtype=np.float64)),
                np.ascontiguousarray(quotes["bid_size"].to_numpy(dtype=np.float64)),
                np.ascontiguousarray(quotes["ask_size"].to_numpy(dtype=np.float64)),
                np.ascontiguousarray(orders["submit_idx"].to_numpy(dtype=np.int64)),
                np.ascontiguousarray(orders["side"].to_numpy(dtype=np.int32)),
                np.ascontiguousarray(orders["qty"].to_numpy(dtype=np.float64)),
                np.ascontiguousarray(orders["arrival_mid"].to_numpy(dtype=np.float64)),
                int(config.latency_events),
                float(config.taker_fee_bps),
                float(config.slippage_bps),
            )
            elapsed = time.perf_counter() - start
            rows.append({
                "kernel": "market_fill_kernel",
                "engine": "cpp_pybind11",
                "repeat": repeat,
                "elapsed_seconds": elapsed,
                "items_per_second": len(orders) / elapsed,
            })

        for repeat in range(config.benchmark_repeats):
            start = time.perf_counter()
            _ = cpp_module.rolling_mean_std(
                np.ascontiguousarray(returns.astype(np.float64)),
                int(config.rolling_window),
            )
            elapsed = time.perf_counter() - start
            rows.append({
                "kernel": "rolling_mean_std",
                "engine": "cpp_pybind11",
                "repeat": repeat,
                "elapsed_seconds": elapsed,
                "items_per_second": len(returns) / elapsed,
            })

        for repeat in range(config.benchmark_repeats):
            start = time.perf_counter()
            _ = cpp_module.risk_metrics(
                np.ascontiguousarray(returns.astype(np.float64))
            )
            elapsed = time.perf_counter() - start
            rows.append({
                "kernel": "risk_metrics",
                "engine": "cpp_pybind11",
                "repeat": repeat,
                "elapsed_seconds": elapsed,
                "items_per_second": len(returns) / elapsed,
            })

    return pd.DataFrame(rows)

benchmark_report = benchmark_pybind11(cpp_module, quotes, orders, returns, config)

benchmark_summary = (
    benchmark_report
    .groupby(["kernel", "engine"])
    .agg(
        mean_elapsed_seconds=("elapsed_seconds", "mean"),
        std_elapsed_seconds=("elapsed_seconds", "std"),
        mean_items_per_second=("items_per_second", "mean"),
    )
    .reset_index()
)

benchmark_summary

In [ ]:
if "cpp_pybind11" in set(benchmark_summary["engine"]):
    plt.figure(figsize=(11, 6))
    labels = []
    values = []
    for _, row in benchmark_summary.iterrows():
        labels.append(row["kernel"] + " / " + row["engine"])
        values.append(row["mean_items_per_second"])
    plt.bar(labels, values)
    plt.title("Python vs C++ pybind11 Throughput")
    plt.ylabel("Items per second")
    plt.xticks(rotation=45, ha="right")
    plt.show()
else:
    benchmark_summary

## 16. Speedup report

Compute speedup by kernel:

$$
speedup = \frac{throughput_{cpp}}{throughput_{python}}
$$

In [ ]:
def compute_speedup_report(benchmark_summary):
    rows = []

    for kernel, sub in benchmark_summary.groupby("kernel"):
        py = sub[sub["engine"] == "python"]
        cpp = sub[sub["engine"] == "cpp_pybind11"]

        if len(py) and len(cpp):
            speedup = cpp["mean_items_per_second"].iloc[0] / py["mean_items_per_second"].iloc[0]
        else:
            speedup = np.nan

        rows.append({
            "kernel": kernel,
            "python_items_per_second": py["mean_items_per_second"].iloc[0] if len(py) else np.nan,
            "cpp_items_per_second": cpp["mean_items_per_second"].iloc[0] if len(cpp) else np.nan,
            "speedup": speedup,
        })

    return pd.DataFrame(rows)

speedup_report = compute_speedup_report(benchmark_summary)

speedup_report

## 17. Memory-layout diagnostics

C++ kernels should receive contiguous arrays.

The wrapper uses `np.ascontiguousarray`, but diagnostics make this explicit.

In [ ]:
memory_layout_report = pd.DataFrame([
    {
        "array": "bid",
        "dtype": str(quotes["bid"].to_numpy().dtype),
        "c_contiguous_raw": quotes["bid"].to_numpy().flags["C_CONTIGUOUS"],
        "c_contiguous_wrapped": np.ascontiguousarray(quotes["bid"].to_numpy()).flags["C_CONTIGUOUS"],
    },
    {
        "array": "submit_idx",
        "dtype": str(orders["submit_idx"].to_numpy().dtype),
        "c_contiguous_raw": orders["submit_idx"].to_numpy().flags["C_CONTIGUOUS"],
        "c_contiguous_wrapped": np.ascontiguousarray(orders["submit_idx"].to_numpy()).flags["C_CONTIGUOUS"],
    },
    {
        "array": "returns",
        "dtype": str(returns.dtype),
        "c_contiguous_raw": returns.flags["C_CONTIGUOUS"],
        "c_contiguous_wrapped": np.ascontiguousarray(returns).flags["C_CONTIGUOUS"],
    },
])

memory_layout_report

## 18. Package layout template

A real repo should put bindings into a stable package structure:

```text
alpha_factory/
  cpp/
    quant_cpp_kernels.cpp
    CMakeLists.txt
  python/
    alpha_factory/
      __init__.py
      execution/
        kernels.py
  tests/
    test_cpp_parity.py
  pyproject.toml
```

Notebook compilation is useful for research, but production should use a proper build system.

In [ ]:
package_layout = pd.DataFrame([
    {"path": "cpp/quant_cpp_kernels.cpp", "purpose": "C++ pybind11 source"},
    {"path": "cpp/CMakeLists.txt", "purpose": "CMake extension build"},
    {"path": "python/alpha_factory/execution/kernels.py", "purpose": "Python wrapper and fallback logic"},
    {"path": "tests/test_cpp_parity.py", "purpose": "Python-vs-C++ parity tests"},
    {"path": "pyproject.toml", "purpose": "Build metadata"},
    {"path": "README.md", "purpose": "Build instructions"},
])

package_layout

## 19. Minimal wrapper pattern

A production wrapper should:

1. try importing C++ module;
2. expose same API as Python fallback;
3. record whether C++ acceleration is active;
4. never silently change numerical results.

In [ ]:
wrapper_template = """
try:
    import quant_cpp_kernels as _cpp
    CPP_AVAILABLE = True
except Exception:
    _cpp = None
    CPP_AVAILABLE = False

def market_fill_kernel(*args, use_cpp=True, **kwargs):
    if use_cpp and CPP_AVAILABLE:
        return _cpp.market_fill_kernel(*args, **kwargs)
    return python_market_fill_kernel(*args, **kwargs)
"""

print(wrapper_template)

## 20. Results table

Summarise build, parity, benchmark, and fallback status.

In [ ]:
cpp_available = cpp_module is not None
parity_passed = bool(parity_report["passed"].all()) if len(parity_report) else False
exception_passed = bool(exception_report["passed"].all()) if len(exception_report) else False

if cpp_available and "cpp_pybind11" in set(benchmark_summary["engine"]):
    mean_speedup = speedup_report["speedup"].dropna().mean()
    min_speedup = speedup_report["speedup"].dropna().min()
else:
    mean_speedup = np.nan
    min_speedup = np.nan

results_summary = pd.DataFrame([{
    "cpp_available": cpp_available,
    "pybind11_available": build_env["pybind11_available"],
    "compiler_available": build_env["compiler"] is not None,
    "extension_path": str(extension_path) if extension_path is not None else None,
    "import_message": import_message,
    "parity_passed": parity_passed,
    "exception_handling_passed": exception_passed,
    "mean_speedup": mean_speedup,
    "min_speedup": min_speedup,
    "fallback_mode": not cpp_available,
}])

results_summary.T

## 21. Governance flags

A C++ binding layer should be reviewed if:

1. pybind11 is unavailable;
2. no compiler is available;
3. extension build fails;
4. import fails;
5. parity tests fail;
6. exception handling fails;
7. C++ speedup is not meaningful;
8. arrays are not contiguous before entering C++;
9. production package layout is missing.

In [ ]:
governance_flags = pd.DataFrame([{
    "cpp_available": cpp_available,
    "pybind11_available": build_env["pybind11_available"],
    "compiler_available": build_env["compiler"] is not None,
    "parity_passed": parity_passed,
    "exception_handling_passed": exception_passed,
    "mean_speedup": mean_speedup,
    "min_speedup": min_speedup,
    "all_wrapped_arrays_contiguous": bool(memory_layout_report["c_contiguous_wrapped"].all()),
    "flag_pybind11_unavailable": bool(not build_env["pybind11_available"]),
    "flag_compiler_unavailable": bool(build_env["compiler"] is None),
    "flag_extension_unavailable": bool(not cpp_available),
    "flag_parity_failed": bool(cpp_available and not parity_passed),
    "flag_exception_handling_failed": bool(cpp_available and not exception_passed),
    "flag_low_speedup": bool(np.isfinite(min_speedup) and min_speedup < 1.5),
    "flag_noncontiguous_wrapped_arrays": bool(not memory_layout_report["c_contiguous_wrapped"].all()),
    "flag_not_packaged_with_cmake_or_pyproject": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_pybind11_unavailable",
        "flag_compiler_unavailable",
        "flag_extension_unavailable",
        "flag_parity_failed",
        "flag_exception_handling_failed",
        "flag_low_speedup",
        "flag_noncontiguous_wrapped_arrays",
        "flag_not_packaged_with_cmake_or_pyproject",
    ]
].any(axis=1)

governance_flags

## 22. Audit checks

Numerical and process checks:

1. Python outputs are finite;
2. C++ outputs are finite if available;
3. parity report exists;
4. build log exists;
5. exception handling report exists;
6. benchmark report exists;
7. governance flags exist.

In [ ]:
audit_rows = []

python_outputs_finite = bool(
    np.isfinite(py_fills["fill_price"]).all()
    and np.isfinite(py_fills["fill_qty"]).all()
    and np.isfinite(py_fills["fee"]).all()
    and np.isfinite(py_fills["shortfall_bps"]).all()
    and np.isfinite(py_risk).all()
)
audit_rows.append({
    "check": "python_outputs_finite",
    "value": python_outputs_finite,
    "passed": python_outputs_finite,
})

if cpp_fills is not None:
    cpp_outputs_finite = bool(
        np.isfinite(np.asarray(cpp_fills["fill_price"])).all()
        and np.isfinite(np.asarray(cpp_fills["fill_qty"])).all()
        and np.isfinite(np.asarray(cpp_fills["fee"])).all()
        and np.isfinite(np.asarray(cpp_fills["shortfall_bps"])).all()
        and np.isfinite(np.asarray(cpp_risk)).all()
    )
else:
    cpp_outputs_finite = False

audit_rows.append({
    "check": "cpp_outputs_finite_if_available",
    "value": cpp_outputs_finite,
    "passed": bool(cpp_outputs_finite or not cpp_available),
})

parity_exists = bool(len(parity_report) > 0)
audit_rows.append({
    "check": "parity_report_exists",
    "value": parity_exists,
    "passed": parity_exists,
})

build_log_exists = build_log is not None
audit_rows.append({
    "check": "build_log_exists",
    "value": build_log_exists,
    "passed": build_log_exists,
})

exception_report_exists = bool(len(exception_report) > 0)
audit_rows.append({
    "check": "exception_report_exists",
    "value": exception_report_exists,
    "passed": exception_report_exists,
})

benchmark_exists = bool(len(benchmark_report) > 0)
audit_rows.append({
    "check": "benchmark_report_exists",
    "value": benchmark_exists,
    "passed": benchmark_exists,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 23. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

quotes.to_csv(output_dir / "quotes.csv", index=False)
orders.to_csv(output_dir / "orders.csv", index=False)
pd.DataFrame(py_fills).to_csv(output_dir / "python_market_fills.csv", index=False)
pd.DataFrame(py_rolling).to_csv(output_dir / "python_rolling_stats.csv", index=False)
pd.Series(py_risk, index=["mean", "vol", "downside_vol", "max_drawdown", "hit_rate"]).to_frame("value").to_csv(output_dir / "python_risk_metrics.csv")

if cpp_fills is not None:
    pd.DataFrame({k: np.asarray(v) for k, v in cpp_fills.items()}).to_csv(output_dir / "cpp_market_fills.csv", index=False)
    pd.DataFrame({k: np.asarray(v) for k, v in cpp_rolling.items()}).to_csv(output_dir / "cpp_rolling_stats.csv", index=False)
    pd.Series(np.asarray(cpp_risk), index=["mean", "vol", "downside_vol", "max_drawdown", "hit_rate"]).to_frame("value").to_csv(output_dir / "cpp_risk_metrics.csv")

parity_report.to_csv(output_dir / "parity_report.csv", index=False)
exception_report.to_csv(output_dir / "exception_report.csv", index=False)
benchmark_report.to_csv(output_dir / "benchmark_report.csv", index=False)
benchmark_summary.to_csv(output_dir / "benchmark_summary.csv", index=False)
speedup_report.to_csv(output_dir / "speedup_report.csv", index=False)
memory_layout_report.to_csv(output_dir / "memory_layout_report.csv", index=False)
package_layout.to_csv(output_dir / "package_layout_template.csv", index=False)
results_summary.to_csv(output_dir / "results_summary.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

build_env_path = output_dir / "build_environment.json"
build_env_path.write_text(json.dumps(build_env, indent=2, default=str), encoding="utf-8")

build_log_path = output_dir / "build_log.json"
build_log_path.write_text(json.dumps(build_log, indent=2, default=str), encoding="utf-8")

manifest = {
    "dataset_name": "cpp_python_bindings_pybind11_outputs",
    "schema_version": "cpp_python_bindings_pybind11_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "cpp_path": str(cpp_path),
    "extension_path": str(extension_path) if extension_path is not None else None,
    "build_environment": build_env,
    "cpp_available": cpp_available,
    "import_message": import_message,
    "results_summary": results_summary.to_dict(orient="records"),
    "parity_report": parity_report.to_dict(orient="records"),
    "speedup_report": speedup_report.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook demonstrates pybind11 bindings for finance/execution kernels.",
        "Kernels include market fills, rolling statistics and risk metrics.",
        "Python reference implementations are used for parity testing.",
        "If pybind11 or a compiler is unavailable, the notebook falls back gracefully to Python-only mode.",
        "Production packaging should use CMake, scikit-build-core, setuptools or a pyproject-based build system."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "parity_report.csv", output_dir / "benchmark_summary.csv", output_dir / "governance_flags.csv", manifest_path

## 24. Practical checklist for pybind11 finance kernels

Before trusting a binding layer:

1. **Write the Python reference first.**
2. **Keep C++ kernels small and deterministic.**
3. **Use contiguous NumPy arrays.**
4. **Validate shapes and dtypes in C++.**
5. **Expose clear Python exceptions.**
6. **Run parity tests on every build.**
7. **Benchmark representative workloads.**
8. **Record build environment.**
9. **Use a real packaging system.**
10. **Never silently fall back without logging it.**

## 25. Limitations

### Build environment dependence

This notebook needs a C++ compiler and `pybind11`. Some notebook environments may not have them.

### Notebook build only

The extension is compiled from inside the notebook. Production should use `pyproject.toml`, CMake, or scikit-build-core.

### Small kernel set

The kernels are intentionally simple. Real systems need richer execution and risk APIs.

### No ABI/version management

Production packages should pin compiler flags, Python versions, NumPy ABI expectations, and platform wheels.

### No continuous integration

Real C++ bindings should be tested in CI across operating systems and Python versions.

### No threading/GIL release

These kernels do not release the GIL. Long-running kernels should consider `py::gil_scoped_release`.

## 26. Research frontier and extensions

Important extensions include:

1. CMake-based build system;
2. scikit-build-core packaging;
3. CI parity tests;
4. GIL release for long kernels;
5. OpenMP parallel loops;
6. SIMD-friendly array layouts;
7. C++ classes for order books;
8. pybind11 wrappers for execution engines;
9. pybind11 plus Arrow/Parquet interfaces;
10. integration with the microstructure friction C++ core.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_12_l1_bid_ask_backtest_execution_model**  
   Use C++ kernels inside an L1 execution backtest.

2. **06_13_production_reconciliation_and_breaks**  
   Reconcile Python/C++ outputs with production records.

3. **Phase 7 execution capstone**  
   Package C++ kernels into a reusable execution research library.

## 28. Summary

This notebook implemented C++/Python bindings with pybind11.

It showed how to:

1. design Python reference kernels;
2. write C++ kernels with NumPy array inputs;
3. expose C++ functions as a Python module;
4. compile a pybind11 extension from a notebook;
5. import the compiled module;
6. validate Python/C++ parity;
7. test C++ exception handling;
8. benchmark Python versus C++;
9. inspect memory layout;
10. define a package layout;
11. build fallback logic;
12. create governance flags and audit checks;
13. save outputs and manifests.

The key computational takeaway:

> pybind11 is a clean bridge for moving bottleneck quant kernels from Python into C++ without abandoning Python research workflows.

The key financial takeaway:

> Fast kernels are only useful if their outputs are numerically identical to the transparent reference model.

## 29. Further reading

- pybind11 documentation.
- NumPy C API and buffer protocol documentation.
- scikit-build-core documentation.
- CMake documentation.
- High-performance Python/C++ interoperability patterns.
- Production quant-engineering practices for parity testing and CI.